<a href="https://colab.research.google.com/github/KukuaEshun/Lab/blob/main/Labb4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [2]:
# A simple helper function I will reuse for the whole lab.

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Try it once with a simple question.
question = "What is a microfinance loan? Answer in one short sentence."
answer = ask_llm(question)
print("Answer:", answer)

# Call the API again the "raw" way so I can see how many tokens it used.
raw_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
print("\nToken usage:", raw_response.usage)

Answer: A microfinance loan is a small, short-term loan provided to low-income individuals or small businesses, often with more flexible repayment terms.

Token usage: CompletionUsage(completion_tokens=28, prompt_tokens=48, total_tokens=76, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.062872872, prompt_time=0.003668058, completion_time=0.059893657, total_time=0.063561715)


The system role sets the AI's instructions for the whole conversation like telling it what job to do and what rules to follow. It doesn't change.

The user role is the actual question or request you're sending right now it changes every time.

Example: In my extraction task, the system prompt said "You are a data extraction assistant... return only JSON with these keys." The user prompt was the actual letter text I wanted it to read.

A token is roughly a piece of a word   sometimes a whole word, sometimes just part of one. Providers charge per token instead of per request because longer text takes more work for the model to process. A short question is cheap, a long letter with a long answer costs more. Charging per token makes the price match the actual amount of work done.


In [3]:
# Ask the same question 5 times at temperature=0.0 and 5 times at temperature=1.2.

question = "Suggest a name for a savings product for market traders in Accra."

print("=== Temperature = 0.0 ===")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i + 1}. {answer}")

print("\n=== Temperature = 1.2 ===")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i + 1}. {answer}")

=== Temperature = 0.0 ===
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which could appeal to market traders.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Mobi**: This name incorporates "mobi," short for mobile, which could suggest a convenient and accessible savings product for market traders who are always on the go.
5. **Kokroko Savings**: "Kokroko" is a Ghanaian word that means "honest" or "trustworthy." This name could convey a sense of reliability and security, which is important for a savings product.
6. **Adanfo Account**: "Adanfo" means "friends" or "partners" in the Akan language. This name

At temperature 0.0, I got nearly the same answer every time I asked. At temperature 1.2, the answers had   different wording, different ideas each time.

For the loan system, I'd use a low temperature (close to 0). This system is dealing with real loan amounts and applicant info that a human officer will actually rely on, so I need the same letter to give the same answer every time.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [5]:
# --- V1: a naive, simple prompt ---
SUMMARY_PROMPT_V1 = "Summarize this:"

print("========== V1 RESULTS ==========")
for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    full_prompt = f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"
    summary_v1 = ask_llm(full_prompt, temperature=0)
    print(f"--- {letter_id} (V1) ---")
    print(summary_v1)
    print()

# --- V2: a proper prompt template with a role and clear rules ---
SUMMARY_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer.
Summarize loan application letters in 3-4 sentences.
Be factual and neutral.
Only use information that is actually stated in the letter.
Do not invent, guess, or add any detail that is not in the letter."""

def summarize_letter(letter_text):
    user_prompt = f"Summarize this loan application:\n\n{letter_text}"
    return ask_llm(user_prompt, system_prompt=SUMMARY_SYSTEM_PROMPT, temperature=0)

print("========== V2 RESULTS ==========")
for letter_id in ["L002", "L006"]:
    summary_v2 = summarize_letter(LETTERS[letter_id])
    print(f"--- {letter_id} (V2) ---")
    print(summary_v2)
    print()

========== V1 RESULTS ==========
--- L002 (V1) ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season, and is willing to repay the loan as soon as possible, despite not having collateral at the moment.

--- L006 (V1) ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.

========== V2 RESULTS ==========
--- L002 (V2) ---
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his trotro engine and settle pers

Here's the short version:

1. V1 gave long, rambling summaries that just retold the letter instead of condensing it, with no consistent length. V2 fixed this by giving it a role and rules (3-4 sentences, factual, no extra details), which made the summaries shorter and more consistent.

2. "No invented details" matters because this system helps make real decisions about people's loans  if the AI makes up a number or detail, a loan officer could act on false information. This is called hallucination in LLM literature  when a model confidently generates information that isn't actually true or in the source text.
